# Phase 3.2: Feature Selection

**Tujuan:**  
Memilih fitur terbaik dari hasil Phase 3.1 (Derived Features) untuk klasifikasi emosi.
Fitur yang redundan, konstan, atau tidak informatif akan dibuang agar model ML
bekerja lebih efisien dan tidak overfitting.

**Pipeline Feature Selection:**
1. **Buang Zero-Variance** — fitur yang nilainya konstan (std ≈ 0)
2. **Buang Multikolinear** — fitur yang berkorelasi sangat tinggi (|r| > 0.95)
3. **F-Score (ANOVA F-value)** — seberapa berbeda nilai fitur antar kelas emosi
4. **Mutual Information** — seberapa banyak informasi fitur tentang kelas emosi
5. **Random Forest Importance** — seberapa penting fitur menurut model tree-based
6. **Ranking gabungan** — rata-rata ranking dari 3 metode → pilih Top-K fitur

**Input:** CSV dari Phase 3.1 (`engineered_features_*.csv`)  
**Output:** CSV fitur terpilih per metode + ranking

> **PERINGATAN LEAKAGE:** Ranking/seleksi fitur di notebook ini dihitung memakai
> label `class` dari SELURUH 675 baris (semua subjek, tidak di-split train/test).
> File `selected_top10_*.csv` / `selected_significant_*.csv` di sini HANYA untuk
> eksplorasi & pelaporan (mis. "fitur mana yang paling diskriminatif") -- JANGAN
> dipakai langsung sebagai input classifier (train+test) karena itu data leakage
> (fitur sudah "melihat" label test set saat diseleksi). Untuk klasifikasi,
> gunakan `engineered_features_*.csv` (Phase 3.1) dan biarkan Phase 4 melakukan
> feature selection (SelectKBest) DI DALAM Pipeline per training-fold saja.

---

In [ ]:
# ============================================================
# SEL INI: IMPORT LIBRARY & KONFIGURASI
# ------------------------------------------------------------
# Memuat library (pandas, sklearn, seaborn), mendefinisikan
# path input (Phase 3.1 CSV) dan output (CSV fitur terpilih).
# Memuat semua 7 CSV fitur dari Phase 3.1.
#
# PENTING:
#   CLASS_COL = 'class_label' -> teks ('negative','neutral','positive')
#   TARGET    = 'class'       -> numerik (0,1,2), dipakai sbg y di sklearn
#   ID_COLS   = kolom identitas yang BUKAN fitur
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.facecolor'] = 'white'

# === PATH ===
FEAT_DIR   = Path(r'D:\Skripsi\new_data\phase_3_feature_engineering\csv')
OUTPUT_DIR = Path(r'D:\Skripsi\new_data\phase_3_feature_engineering\3.2_feature_selection\output')
FIG_DIR    = Path(r'D:\Skripsi\new_data\phase_3_feature_engineering\3.2_feature_selection\figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_COL = 'class_label'
TARGET    = 'class'
EMOTIONS  = ['negative', 'neutral', 'positive']
PALETTE   = {'negative': '#e74c3c', 'neutral': '#3498db', 'positive': '#2ecc71'}

# Kolom identitas (BUKAN fitur)
ID_COLS = ['subject_id', 'subject_num', 'session', 'trial', 'class', 'class_label']

DATASETS = {
    'GC'           : FEAT_DIR / 'engineered_features_gc.csv',
    'PDC Delta'    : FEAT_DIR / 'engineered_features_pdc_delta.csv',
    'PDC Theta'    : FEAT_DIR / 'engineered_features_pdc_theta.csv',
    'PDC Alpha'    : FEAT_DIR / 'engineered_features_pdc_alpha.csv',
    'PDC Beta'     : FEAT_DIR / 'engineered_features_pdc_beta.csv',
    'PDC Gamma'    : FEAT_DIR / 'engineered_features_pdc_gamma.csv',
    'PDC Broadband': FEAT_DIR / 'engineered_features_pdc_broadband.csv',
}

data = {}
for name, csv_path in DATASETS.items():
    if not csv_path.exists():
        print(f'  [SKIP] {name}: belum ada. Jalankan Phase 3.1 terlebih dahulu.')
        continue
    df = pd.read_csv(csv_path)
    data[name] = df
    n_feat = len([c for c in df.columns if c not in ID_COLS
                  and df[c].dtype in ['float64','int64']])
    print(f'  Loaded {name}: {df.shape[0]} baris, {n_feat} fitur numerik')

print(f'\nOutput : {OUTPUT_DIR}')
print(f'Figures: {FIG_DIR}')


---
## Step 1: Buang Fitur Zero-Variance

Fitur dengan standar deviasi ≈ 0 (konstan) tidak memberikan informasi apapun
untuk klasifikasi dan HARUS dibuang sebelum memasukkan data ke model ML.

In [ ]:
# ============================================================
# SEL INI: BUANG FITUR ZERO-VARIANCE
# ------------------------------------------------------------
# Mengidentifikasi dan membuang kolom dengan std < 1e-10.
# Fitur ini mencakup:
#   - PDC Degree (mean_out/in/total_degree) → selalu 61/62/122
#   - DI_degree, DI_strength, ratio_degree, ratio_strength
#     → selalu 0 atau 1 karena out == in secara matematis
#   - Z-score turunan dari fitur konstan di atas
#
# Hasil: dictionary 'data_clean' berisi DataFrame tanpa
#        kolom zero-variance.
# ============================================================
data_clean = {}

for name, df in data.items():
    # Identifikasi fitur numerik
    feat_cols = [c for c in df.columns if c not in ID_COLS
                 and df[c].dtype in ['float64', 'int64']]

    # Cari fitur zero-variance
    zero_var = [c for c in feat_cols if df[c].std() < 1e-10]
    keep     = [c for c in feat_cols if df[c].std() >= 1e-10]

    # Simpan versi bersih (ID cols + fitur non-konstan)
    data_clean[name] = df[ID_COLS + keep].copy()

    print(f'\n  {name}:')
    print(f'    Sebelum : {len(feat_cols)} fitur')
    print(f'    Dibuang : {len(zero_var)} zero-variance')
    print(f'    Tersisa : {len(keep)} fitur')
    if zero_var:
        print(f'    Dibuang : {zero_var}')


---
## Step 2: Buang Fitur Multikolinear (|r| > 0.95)

Fitur yang berkorelasi sangat tinggi mengandung informasi yang redundan.
Jika dua fitur memiliki |r| > 0.95, salah satu dibuang (yang memiliki
rata-rata korelasi lebih tinggi dengan fitur lain).

In [ ]:
# ============================================================
# SEL INI: BUANG FITUR MULTIKOLINEAR (|r| > 0.95)
# ------------------------------------------------------------
# Untuk setiap pasangan fitur dengan korelasi Pearson > 0.95,
# buang fitur yang memiliki rata-rata korelasi absolut lebih
# tinggi terhadap semua fitur lain (lebih redundan).
#
# Contoh: jika mean_out_strength dan mean_in_strength
# berkorelasi 0.99, salah satu akan dibuang.
#
# Threshold 0.95 dipilih karena cukup ketat untuk menghindari
# multikolinearitas tanpa kehilangan informasi yang beragam.
# ============================================================
CORR_THRESHOLD = 0.95

data_reduced = {}

for name, df in data_clean.items():
    feat_cols = [c for c in df.columns if c not in ID_COLS]
    corr_mat  = df[feat_cols].corr().abs()

    # Tandai fitur yang perlu dibuang
    to_drop = set()
    for i in range(len(feat_cols)):
        for j in range(i+1, len(feat_cols)):
            if corr_mat.iloc[i, j] > CORR_THRESHOLD:
                # Buang yang punya rata-rata korelasi lebih tinggi
                mean_i = corr_mat.iloc[i].drop(feat_cols[i]).mean()
                mean_j = corr_mat.iloc[j].drop(feat_cols[j]).mean()
                drop_col = feat_cols[i] if mean_i > mean_j else feat_cols[j]
                to_drop.add(drop_col)

    keep = [c for c in feat_cols if c not in to_drop]
    data_reduced[name] = df[ID_COLS + keep].copy()

    print(f'\n  {name}:')
    print(f'    Sebelum : {len(feat_cols)} fitur')
    print(f'    Dibuang : {len(to_drop)} multikolinear (|r|>{CORR_THRESHOLD})')
    print(f'    Tersisa : {len(keep)} fitur')
    if to_drop:
        print(f'    Dibuang : {sorted(to_drop)}')


---
## Step 3: Feature Ranking — 3 Metode

Tiga metode scoring digunakan untuk meranking setiap fitur:

| Metode | Cara Kerja | Kekuatan |
|--------|------------|----------|
| **F-Score** (ANOVA) | Rasio varians antar-kelas vs dalam-kelas | Cepat, menangkap perbedaan mean |
| **Mutual Information** | Informasi bersama antara fitur dan target | Menangkap hubungan non-linear |
| **Random Forest** | Importance dari ensemble decision trees | Menangkap interaksi antar fitur |

In [ ]:
# ============================================================
# SEL INI: FEATURE RANKING DENGAN 3 METODE
# ------------------------------------------------------------
# Menghitung skor setiap fitur menggunakan 3 metode:
#   1) F-Score (ANOVA F-value) — sklearn.f_classif
#   2) Mutual Information     — sklearn.mutual_info_classif
#   3) Random Forest Importance — sklearn.RandomForestClassifier
#
# Setiap metode menghasilkan ranking (1 = terbaik).
# Ranking gabungan = rata-rata ranking dari 3 metode.
#
# Target (y) = kolom 'class' (numerik 0/1/2).
# Fitur (X) = semua kolom numerik setelah filtering step 1-2.
# Data di-standardisasi (z-score global) sebelum scoring.
# ============================================================
all_rankings = {}

for name, df in data_reduced.items():
    feat_cols = [c for c in df.columns if c not in ID_COLS]
    X = df[feat_cols].values
    y = df[TARGET].values

    # Standardisasi
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 1. F-Score (ANOVA)
    f_scores, f_pvals = f_classif(X_scaled, y)

    # 2. Mutual Information (rata-rata 5 run untuk stabilitas)
    mi_scores = np.mean([
        mutual_info_classif(X_scaled, y, random_state=seed)
        for seed in range(5)
    ], axis=0)

    # 3. Random Forest Importance
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10,
        random_state=42, n_jobs=-1,
    )
    rf.fit(X_scaled, y)
    rf_scores = rf.feature_importances_

    # Buat tabel ranking
    ranking = pd.DataFrame({
        'feature'     : feat_cols,
        'f_score'     : f_scores,
        'f_pvalue'    : f_pvals,
        'mi_score'    : mi_scores,
        'rf_importance': rf_scores,
    })

    # Ranking (1 = terbaik = skor tertinggi)
    ranking['rank_f']  = ranking['f_score'].rank(ascending=False).astype(int)
    ranking['rank_mi'] = ranking['mi_score'].rank(ascending=False).astype(int)
    ranking['rank_rf'] = ranking['rf_importance'].rank(ascending=False).astype(int)
    ranking['rank_avg'] = ((ranking['rank_f'] + ranking['rank_mi'] + ranking['rank_rf']) / 3).round(2)

    # Urutkan berdasarkan ranking gabungan
    ranking = ranking.sort_values('rank_avg').reset_index(drop=True)
    all_rankings[name] = ranking

    # Simpan
    ranking.to_csv(OUTPUT_DIR / f'feature_ranking_{name.lower().replace(" ","_")}.csv', index=False)

    print(f'\n{"="*75}')
    print(f'  {name} — Feature Ranking (Top 10)')
    print(f'{"="*75}')
    print(f'  {"#":<3} {"Feature":<28} {"F-Score":>10} {"MI":>10} {"RF Imp":>10} {"Avg Rank":>10}')
    print('  ' + '-'*70)
    for i, row in ranking.head(10).iterrows():
        print(f'  {i+1:<3} {row["feature"]:<28} {row["f_score"]:>10.2f} '
              f'{row["mi_score"]:>10.4f} {row["rf_importance"]:>10.4f} {row["rank_avg"]:>10.1f}')


---
## Step 4: Visualisasi Feature Importance

Horizontal bar chart menunjukkan skor setiap fitur dari 3 metode.
Fitur diurutkan berdasarkan ranking gabungan (terbaik di atas).

In [ ]:
# ============================================================
# SEL INI: BAR CHART FEATURE IMPORTANCE
# ------------------------------------------------------------
# Membuat 3 subplot per metode:
#   1) F-Score (biru)
#   2) Mutual Information (hijau)
#   3) Random Forest Importance (oranye)
# Fitur diurutkan berdasarkan ranking gabungan (terbaik di atas).
#
# Satu gambar per metode, disimpan ke figures/.
# Output: {method}_feature_importance.png
# ============================================================
for name, ranking in all_rankings.items():
    # Ambil semua fitur, urutkan berdasarkan rank_avg
    plot_df = ranking.sort_values('rank_avg', ascending=True).copy()
    n_feat  = len(plot_df)

    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, n_feat * 0.35)))

    for ax, score_col, title, color in [
        (axes[0], 'f_score',      'F-Score (ANOVA)',        '#3498db'),
        (axes[1], 'mi_score',     'Mutual Information',     '#2ecc71'),
        (axes[2], 'rf_importance','Random Forest Importance','#e67e22'),
    ]:
        # Normalisasi ke [0,1] untuk perbandingan visual
        vals = plot_df[score_col].values
        if vals.max() > 0:
            vals_norm = vals / vals.max()
        else:
            vals_norm = vals
        ax.barh(range(n_feat), vals_norm, color=color, alpha=0.8)
        ax.set_yticks(range(n_feat))
        ax.set_yticklabels(plot_df['feature'].values, fontsize=8)
        ax.set_xlabel('Normalized Score')
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.invert_yaxis()  # terbaik di atas

    fig.suptitle(f'{name} — Feature Importance Comparison',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    p = FIG_DIR / f'{name.lower().replace(" ","_")}_feature_importance.png'
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {p}')
    plt.show()


---
## Step 5: Pilih Top-K Fitur & Simpan

Berdasarkan ranking gabungan, pilih **Top-K fitur terbaik** untuk setiap metode.
Dua kriteria digunakan:
- **Top-K fixed** (K=10) — jumlah fitur tetap
- **F-Score signifikan** (p < 0.05) — hanya fitur yang secara statistik
  berbeda antar kelas emosi

Kedua hasil disimpan sebagai CSV terpisah.

In [ ]:
# ============================================================
# SEL INI: PILIH TOP-K FITUR & SIMPAN CSV
# ------------------------------------------------------------
# Dua strategi seleksi:
#   1) Top-K (K=10): ambil 10 fitur dengan rank_avg terkecil
#   2) Signifikan: ambil fitur dengan F-Score p < 0.05
#
# Output per metode:
#   - selected_top10_{method}.csv    (dataset dengan 10 fitur terbaik)
#   - selected_significant_{method}.csv (dataset dengan fitur signifikan)
#   - feature_ranking_{method}.csv   (sudah disimpan di Step 3)
#
# Kolom ID (subject_id, session, trial, class, class_label)
# tetap dipertahankan untuk traceability.
# ============================================================
TOP_K = 10

CSV_OUT = Path(r'D:\Skripsi\new_data\phase_3_feature_engineering\csv')
CSV_OUT.mkdir(parents=True, exist_ok=True)

for name, ranking in all_rankings.items():
    df = data_reduced[name]
    prefix = name.lower().replace(' ', '_')

    # === Strategy 1: Top-K ===
    k = min(TOP_K, len(ranking))
    top_feats = ranking.head(k)['feature'].tolist()
    df_top = df[ID_COLS + top_feats].copy()
    fname_top = f'selected_top{k}_{prefix}.csv'
    df_top.to_csv(OUTPUT_DIR / fname_top, index=False)
    df_top.to_csv(CSV_OUT / fname_top, index=False)

    # === Strategy 2: F-Score significant (p < 0.05) ===
    sig_feats = ranking[ranking['f_pvalue'] < 0.05]['feature'].tolist()
    if sig_feats:
        df_sig = df[ID_COLS + sig_feats].copy()
    else:
        df_sig = df[ID_COLS].copy()  # tidak ada fitur signifikan
    fname_sig = f'selected_significant_{prefix}.csv'
    df_sig.to_csv(OUTPUT_DIR / fname_sig, index=False)
    df_sig.to_csv(CSV_OUT / fname_sig, index=False)

    print(f'\n  {name}:')
    print(f'    Top-{k} fitur     : {top_feats}')
    print(f'    Signifikan (p<0.05): {len(sig_feats)} fitur')
    print(f'    -> {fname_top}')
    print(f'    -> {fname_sig}')


---
## Step 6: Heatmap Ranking Perbandingan Antar Metode

Heatmap menampilkan ranking gabungan (rank_avg) setiap fitur
untuk semua 7 metode sekaligus. Warna gelap = ranking tinggi (penting).

In [ ]:
# ============================================================
# SEL INI: HEATMAP RANKING LINTAS METODE
# ------------------------------------------------------------
# Membuat satu heatmap besar yang membandingkan ranking
# gabungan (rank_avg) setiap fitur lintas semua 7 metode.
# Fitur yang konsisten penting (ranking tinggi di banyak metode)
# adalah kandidat utama untuk final feature set.
#
# Warna kuning/terang = ranking rendah (penting)
# Warna biru/gelap    = ranking tinggi (kurang penting)
# NaN (kotak kosong)  = fitur tidak ada di metode tersebut
# Output: cross_method_ranking_heatmap.png
# ============================================================
# Kumpulkan semua fitur unik
all_features = set()
for ranking in all_rankings.values():
    all_features.update(ranking['feature'].tolist())

# Buat tabel pivot
heat_rows = []
for name, ranking in all_rankings.items():
    for _, row in ranking.iterrows():
        heat_rows.append({
            'method' : name,
            'feature': row['feature'],
            'rank'   : row['rank_avg'],
        })

heat_df = pd.DataFrame(heat_rows)
pivot   = heat_df.pivot_table(index='feature', columns='method',
                              values='rank', aggfunc='first')

# Urutkan fitur berdasarkan rata-rata ranking lintas metode
pivot['avg_rank'] = pivot.mean(axis=1)
pivot = pivot.sort_values('avg_rank')
pivot = pivot.drop(columns='avg_rank')

# Reorder kolom
method_order = [n for n in DATASETS.keys() if n in pivot.columns]
pivot = pivot[method_order]

fig, ax = plt.subplots(figsize=(max(12, len(method_order)*1.5),
                                max(8, len(pivot)*0.4)))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd_r',
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Ranking Across All Methods\n(lower = more important)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Method')
ax.set_ylabel('Feature')
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
p = FIG_DIR / 'cross_method_ranking_heatmap.png'
fig.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
print(f'Saved: {p}')
plt.show()


---
## 7. Ringkasan Akhir

In [ ]:
# ============================================================
# SEL INI: RINGKASAN AKHIR PHASE 3.2
# ------------------------------------------------------------
# Menampilkan daftar file output, jumlah fitur per tahap
# filtering, dan catatan penting untuk Phase 3.3 dan Phase 4.
# ============================================================
print('='*65)
print('PHASE 3.2 COMPLETE — Feature Selection')
print('='*65)

print(f'\n--- PIPELINE SUMMARY ---')
print(f'  {"Method":<15} {"Awal":>6} {"- ZeroVar":>10} {"- Corr>0.95":>12} {"Top-10":>8} {"Sig(p<0.05)":>12}')
print('  ' + '-'*65)
for name in data.keys():
    n_raw   = len([c for c in data[name].columns if c not in ID_COLS
                   and data[name][c].dtype in ['float64','int64']])
    n_clean = len([c for c in data_clean[name].columns if c not in ID_COLS])
    n_red   = len([c for c in data_reduced[name].columns if c not in ID_COLS])
    n_sig   = len(all_rankings[name][all_rankings[name]['f_pvalue'] < 0.05])
    print(f'  {name:<15} {n_raw:>6} {n_clean:>10} {n_red:>12} {min(10,n_red):>8} {n_sig:>12}')

print(f'\n--- OUTPUT FILES ---')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'  {f.name}')
print(f'\n--- FIGURES ---')
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name}')

print(f'\n--- CATATAN UNTUK PHASE 3.3 & PHASE 4 ---')
print('  [1] Gunakan selected_top10_*.csv untuk klasifikasi awal')
print('  [2] Gunakan selected_significant_*.csv untuk analisis ilmiah')
print('  [3] Phase 3.3 akan membandingkan informativeness GC vs PDC')
print('  [4] Phase 4 akan menggunakan fitur terpilih untuk klasifikasi ML')
